In [1]:
import requests
from bs4 import BeautifulSoup
import re
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

session = requests.Session()
session.headers.update(HEADERS)

# ============== FONCTIONS UTILITAIRES ==============

def clean_text(text):
    if text is None:
        return None
    return re.sub(r"\s+", " ", text).strip()

def split_lieu(lieu_brut):
    if not lieu_brut:
        return None, None
    parts = lieu_brut.split(" - ", 1)
    ville = parts[0].strip() if len(parts) > 0 else None
    adresse = parts[1].strip() if len(parts) > 1 else None
    return ville, adresse

def parse_dates(raw):
    if not raw:
        return None, None
    raw = clean_text(raw)
    m = re.match(r"Du\s+(.+?)\s+au\s+(.+)", raw, re.IGNORECASE)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    m2 = re.match(r"Le\s+(.+)", raw, re.IGNORECASE)
    if m2:
        return m2.group(1).strip(), None
    return raw, None

def get_event_details(url, session):
    try:
        r = session.get(url, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
    except requests.RequestException:
        return None, None, None, None

    # Date
    date_raw = None
    for div in soup.find_all("div", class_="field-item"):
        if "Date :" in div.get_text():
            b = div.find("b")
            if b:
                date_raw = clean_text(b.get_text(" ", strip=True))
                break
    date_debut, date_fin = parse_dates(date_raw)

    # Website
    site_web = None
    for div in soup.find_all("div", class_="titreb"):
        if "Website" in div.get_text():
            next_div = div.find_next_sibling("div", class_="field-items")
            if next_div:
                a = next_div.find("a")
                if a:
                    site_web = a.get("href")
            break

    # Code postal — prend le 2ème s'il y en a deux
    cp = None
    for div in soup.find_all("div", class_="titreb"):
        if "Adresse" in div.get_text():
            next_div = div.find_next_sibling("div", class_="field-items")
            if next_div:
                texte = next_div.get_text(" ", strip=True)
                tous_les_cp = re.findall(r"\b\d{5}\b", texte)
                if len(tous_les_cp) >= 2:
                    cp = tous_les_cp[1]
                elif len(tous_les_cp) == 1:
                    cp = tous_les_cp[0]
            break

    return date_debut, date_fin, site_web, cp

def scrape_row(row, numero):
    title_div = row.find("div", class_="views-field-title")
    if not title_div:
        return None
    a_titre = title_div.find("a")
    titre = clean_text(a_titre.get_text(" ", strip=True)) if a_titre else None
    lien_relatif = a_titre.get("href") if a_titre else None
    lien = f"https://flanerbouger.fr{lien_relatif}" if lien_relatif and lien_relatif.startswith("/") else lien_relatif

    categorie, lieu = None, None
    info_divs = row.find_all("div", class_="views-field-field-location-taxonomize-terms-location-taxonomize-longname")
    for div in info_divs:
        label_span = div.find("span", class_="field-content")
        label = label_span.get_text(strip=True) if label_span else ""
        if "Catégorie" in label:
            cat_link = div.find("span", class_="views-field-field-tags")
            if cat_link:
                a_cat = cat_link.find("a")
                categorie = clean_text(a_cat.get_text(" ", strip=True)) if a_cat else None
        elif "Lieu" in label:
            value_span = div.find("span", class_="views-field-field-postaladdress-postal-code")
            if value_span:
                value_content = value_span.find("span", class_="field-content")
                lieu = clean_text(value_content.get_text(" ", strip=True)) if value_content else None

    ville_evt, adresse_evt = split_lieu(lieu)

    date_debut, date_fin, site_web, cp = None, None, None, None
    if lien:
        date_debut, date_fin, site_web, cp = get_event_details(lien, session)

    return {
        "theme": categorie,
        "nom_event": titre,
        "adresse": adresse_evt,
        "ville": ville_evt,
        "cp": cp,
        "dep": numero,
        "date_d": date_debut,
        "date_f": date_fin,
        "site_web": site_web,
    }

# ============== SCRAPING PRINCIPAL ==============

all_data = []

departements = {
    "01":"ain","02":"aisne","03":"allier","04":"alpes-de-haute-provence",
    "05":"hautes-alpes","06":"alpes-maritimes","07":"ardeche","08":"ardennes",
    "09":"ariege","10":"aube","11":"aude","12":"aveyron","13":"bouches-du-rhone",
    "14":"calvados","15":"cantal","16":"charente","17":"charente-maritime",
    "18":"cher","19":"correze","2A":"corse-du-Sud","2B":"haute-Corse",
    "21":"cote-d-or","22":"Cotes-d-armor","23":"creuse","24":"dordogne",
    "25":"doubs","26":"drome","27":"eure","28":"eure-et-loir","29":"finistere",
    "30":"gard","31":"haute-garonne","32":"gers","33":"gironde","34":"herault",
    "35":"ille-et-vilaine","36":"indre","37":"indre-et-loire","38":"isere",
    "39":"jura","40":"landes","41":"loir-et-cher","42":"loire","43":"haute-loire",
    "44":"loire-atlantique","45":"loiret","46":"lot","47":"lot-et-garonne",
    "48":"lozere","49":"maine-et-loire","50":"manche","51":"marne",
    "52":"haute-marne","53":"mayenne","54":"meurthe-et-moselle","55":"meuse",
    "56":"morbihan","57":"moselle","58":"nievre","59":"nord","60":"oise",
    "61":"orne","62":"pas-de-calais","63":"puy-de-dome","64":"Pyrenees-Atlantiques",
    "65":"hautes-pyrenees","66":"pyrenees-orientales","67":"bas-rhin","68":"haut-rhin",
    "69":"rhone","70":"haute-saone","71":"saone-et-loire","72":"Sarthe",
    "73":"savoie","74":"haute-savoie","75":"ile-de-france","76":"seine-maritime",
    "77":"seine-et-marne","78":"yvelines","79":"deux-sevres","80":"somme",
    "81":"tarn","82":"tarn-et-garonne","83":"var","84":"vaucluse","85":"vendee",
    "86":"vienne","87":"haute-vienne","88":"vosges","89":"yonne",
    "90":"territoire-de-belfort","91":"essonne","92":"hauts-de-seine",
    "93":"seine-saint-denis","94":"val-de-marne","95":"val-d-oise"
}

for numero, nom in departements.items():
    base_url = f"https://flanerbouger.fr/event/{numero}-event-{nom}"
    print(f"\n--- Département {numero} : {nom} ---")

    try:
        r = session.get(base_url, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print(f"Erreur sur {base_url} : {e}")
        continue

    page_numbers = [int(m.group(1)) for a in soup.find_all("a", href=True)
                    if (m := re.search(r"[?&]page=(\d+)", a["href"]))]
    last_page = max(page_numbers) if page_numbers else 1
    print(f"Pages détectées : {last_page}")

    for page in range(1, last_page + 1):
        url_page = base_url if page == 1 else f"{base_url}?page={page}"

        try:
            r = session.get(url_page, timeout=10)
            soup = BeautifulSoup(r.text, "html.parser")
        except Exception as e:
            print(f"Erreur page {page} : {e}")
            continue

        rows = soup.find_all("div", class_="views-row")
        if not rows:
            print(f"Page {page} vide, arrêt.")
            break

        page_data = []
        with ThreadPoolExecutor(max_workers=10) as executor:
            futures = {executor.submit(scrape_row, row, numero): row for row in rows}
            for future in as_completed(futures):
                try:
                    result = future.result()
                    if result:
                        page_data.append(result)
                except Exception as e:
                    print(f"Erreur sur un événement : {e}")

        print(f"  Page {page} → {len(page_data)} événements")
        all_data.extend(page_data)
        time.sleep(0.5)

# ============== EXPORT ==============

df = pd.DataFrame(all_data)
df["adresse_complete"] = df["adresse"].fillna("") + " " + df["cp"].fillna("") + " " + df["ville"].fillna("")
df.to_csv("dfjade.csv", index=False)
print(f"\n✅ Terminé — {len(df)} événements exportés dans dfjade.csv")


--- Département 01 : ain ---
Pages détectées : 11
  Page 1 → 30 événements
  Page 2 → 30 événements
  Page 3 → 30 événements
  Page 4 → 30 événements
  Page 5 → 30 événements
  Page 6 → 30 événements
  Page 7 → 30 événements
  Page 8 → 30 événements
  Page 9 → 30 événements
  Page 10 → 29 événements
  Page 11 → 0 événements

--- Département 02 : aisne ---
Pages détectées : 10
  Page 1 → 30 événements
  Page 2 → 30 événements
  Page 3 → 30 événements
  Page 4 → 30 événements
  Page 5 → 30 événements
  Page 6 → 30 événements
  Page 7 → 30 événements
  Page 8 → 30 événements
  Page 9 → 30 événements
  Page 10 → 15 événements

--- Département 03 : allier ---
Pages détectées : 10
  Page 1 → 30 événements
  Page 2 → 30 événements
  Page 3 → 30 événements
  Page 4 → 30 événements
  Page 5 → 30 événements
  Page 6 → 30 événements
  Page 7 → 30 événements
  Page 8 → 30 événements
  Page 9 → 22 événements
  Page 10 → 0 événements

--- Département 04 : alpes-de-haute-provence ---
Pages détectées

In [2]:
df.head()

,theme,nom_event,adresse,ville,cp,dep,date_d,date_f,site_web,adresse_complete
0,Marchés,Marché tous les jeudis matins,place Carnot,BELLEGARDE-SUR-VALSERINE,01200,01,02 July 2026,NaN,None,place Carnot 01200 BELLEGARDE-SUR-VALSERINE
1,Marchés,Marché hebdomadaire,Marché hebdomadaire Place de la mairie place e...,VILLENEUVE,01480,01,02 July 2026,NaN,None,Marché hebdomadaire Place de la mairie place e...
2,Marchés,Marché mixte,place du Marché,VONNAS,01540,01,02 July 2026,NaN,None,place du Marché 01540 VONNAS
3,Marchés,Marché alimentaire vendredi de chaque semaine,"quartier gare, devant la gare sncf",AMBERIEU-EN-BUGEY,01500,01,03 July 2026,NaN,None,"quartier gare, devant la gare sncf 01500 AMBER..."
4,Marchés bio,Marché Bio vendredi de chaque semaine,place du village,AMBERIEU-EN-BUGEY,01500,01,03 July 2026,NaN,None,place du village 01500 AMBERIEU-EN-BUGEY


In [3]:
df['adresse'] = df['adresse'].str.replace(r'\s*\d{5}\s*-\s*.+$', '', regex=True).str.strip()

In [4]:
df.head()

,theme,nom_event,adresse,ville,cp,dep,date_d,date_f,site_web,adresse_complete
0,Marchés,Marché tous les jeudis matins,place Carnot,BELLEGARDE-SUR-VALSERINE,01200,01,02 July 2026,NaN,None,place Carnot 01200 BELLEGARDE-SUR-VALSERINE
1,Marchés,Marché hebdomadaire,Marché hebdomadaire Place de la mairie place e...,VILLENEUVE,01480,01,02 July 2026,NaN,None,Marché hebdomadaire Place de la mairie place e...
2,Marchés,Marché mixte,place du Marché,VONNAS,01540,01,02 July 2026,NaN,None,place du Marché 01540 VONNAS
3,Marchés,Marché alimentaire vendredi de chaque semaine,"quartier gare, devant la gare sncf",AMBERIEU-EN-BUGEY,01500,01,03 July 2026,NaN,None,"quartier gare, devant la gare sncf 01500 AMBER..."
4,Marchés bio,Marché Bio vendredi de chaque semaine,place du village,AMBERIEU-EN-BUGEY,01500,01,03 July 2026,NaN,None,place du village 01500 AMBERIEU-EN-BUGEY
